# Every method against the measurement

Notebook 05 built a prediction of $Y_v$ from first principles. This one asks the
only question that matters about it: **is it any good?** — by putting it beside
the empirical formulas the industry actually uses, and beside experiment.

The comparison is on KVLCC2, for which two independent experimental values are
available. That turns out to matter, so it is dealt with first.

## Two measurements, not one

The reference set in this repository is the MMG linear hull derivative set, from
model tests published by Yasukawa and Yoshimura (2015). Chame et al. (2025) quote
a different experimental value for the same hull. Both are "EFD".

| source | $Y'_V$ (MMG normalisation) |
|---|---|
| MMG set, Yasukawa & Yoshimura (2015) | 0.315 |
| Chame et al. (2025), Table 9 | 0.271 |

That is a **16% spread between two measurements of the same ship**. No prediction
method below is going to be judged meaningfully at better than that, and any
claim of agreement inside 16% should be read with it in mind. Reporting a single
"experimental" number would hide this.

In [1]:
using LinearAlgebra
using MarineHydro
using Printf
using TOML

data_directory = joinpath("..", "validation", "gothenburg2010", "data", "KVLCC2")
surface_paths = [joinpath(data_directory, "kvlcc_bow1.dat"),
    joinpath(data_directory, "kvlcc2_stn1.dat")]

# Full-scale KVLCC2 particulars, matching validation/kvlcc2_maneuvering.
kvlcc2 = HullParticulars(; length_pp = 320.0, beam = 58.0, draft = 20.8,
    block_coefficient = 0.8098)
draft_ratio = kvlcc2.draft / kvlcc2.length_pp

reference = TOML.parsefile(joinpath("..", "validation", "kvlcc2_maneuvering",
    "reference.toml"))
mmg_measured = abs(reference["mmg"]["linear_hull"]["Y_v"])
chame_measured = 0.271          # Chame et al. (2025), Table 9, KVLCC2 column
chame_cfd = 0.276               # their RANS value for the same hull
@printf("MMG set   Y'_V = %.3f\nChame EFD Y'_V = %.3f   (spread %.1f%%)\n",
    mmg_measured, chame_measured,
    100 * abs(mmg_measured - chame_measured) / chame_measured)

MMG set   Y'_V = 0.315
Chame EFD Y'_V = 0.271   (spread 16.2%)


## The empirical formulas

All of these are corrections applied to the same theoretical backbone, the
low-aspect-ratio result $Y'_V = \pi T/L$. They are collected in Chame et al.
Table 1, and implemented here in `empirical_derivatives.jl` with a test that
reproduces their published KVLCC2 column to 3%, so a mistyped coefficient shows
up as a test failure rather than as a quietly wrong curve.

In [2]:
labels = Dict(:low_aspect_ratio => "Low aspect ratio (theory)",
    :jacobs => "Jacobs (1966)", :smitt => "Smitt (1970)",
    :norrbin => "Norrbin (1971)", :inoue => "Inoue et al. (1981)",
    :clarke => "Clarke, Gedling & Hine (1982)",
    :ho_young_lee => "Ho-Young Lee et al. (1998)")

println(rpad("method", 34), rpad("Y'_V", 10), rpad("vs Chame EFD", 15),
    "vs MMG set")
for method in empirical_methods()
    value = empirical_sway_derivatives(kvlcc2; method).Y_v
    @printf("%-34s%-10.4f%+-15.1f%+.1f%%\n", labels[method], value,
        100 * (value / chame_measured - 1), 100 * (value / mmg_measured - 1))
end
@printf("\n%-34s%-10.4f\n", "EFD (Chame et al. 2025)", chame_measured)
@printf("%-34s%-10.4f\n", "EFD (MMG set)", mmg_measured)
@printf("%-34s%-10.4f\n", "RANS (Chame et al. 2025)", chame_cfd)

method                            Y'_V      vs Chame EFD   vs MMG set


Low aspect ratio (theory)         0.2042    -24.6          -35.2%
Jacobs (1966)                     0.2042    -24.6          -35.2%
Smitt (1970)                      0.3247    +19.8          +3.1%
Norrbin (1971)                    0.3568    +31.7          +13.3%
Inoue et al. (1981)               0.4097    +51.2          +30.1%
Clarke, Gedling & Hine (1982)     0.3886    +43.4          +23.4%
Ho-Young Lee et al. (1998)        0.3077    +13.6          -2.3%

EFD (Chame et al. 2025)           0.2710    
EFD (MMG set)                     0.3150    
RANS (Chame et al. 2025)          0.2760    


Two things stand out, and they are the whole reason Chame et al. wrote their
paper.

The regressions **split into two camps**. Those fitted before about 1980 — Jacobs
and the bare low-aspect-ratio theory — come out *below* the measurement. Those
fitted later — Norrbin, Inoue, Clarke — come out well *above* it, by 30 to 50%.
Clarke, the most widely used of them, over-predicts this hull by more than 40%
against Chame's measurement.

That is not a defect of any one formula. It is that the databases behind them
were assembled from hulls of a different era, and KVLCC2 is a modern full-form
tanker. Chame et al.'s response was to regenerate the database: 690 virtual
static drift tests across 115 parametrically generated modern hulls, and a fresh
regression for $Y'_V$ carrying explicit bow and stern shape terms. They report a
maximum relative error of 4.7% on $Y'_V$, and a reduction in trajectory error
from 23% to 10% when the new coefficient is used in a maneuvering simulation.

## Where this package's method lands

The shed-vorticity model of notebook 05 is not a regression. It is a slender-body
momentum balance terminated at a separation station that the boundary layer
predicts, plus the displacement effect of that layer. Nothing in it is fitted to
maneuvering data.

So the interesting question is not whether it beats a regression fitted to
hundreds of hulls, but *where it falls* — and in particular whether it behaves
like the theory it is built on or like the data-fitted formulas.

In [3]:
forward_speed = 0.142 * sqrt(SETTINGS.g)
kinematic_viscosity = forward_speed / 4.6e6
WATER_DENSITY = 1025.0

grid = read_gothenburg2010_panel_grid(surface_paths; target_shape = (16, 9))
mesh = grid.mesh
ship_length = maximum(mesh.centers[:, 1]) - minimum(mesh.centers[:, 1])
midship = (maximum(mesh.centers[:, 1]) + minimum(mesh.centers[:, 1])) / 2

surge = solve_rigid_body_potential(mesh, :surge)
sway = solve_rigid_body_potential(mesh, :sway)
yaw = solve_rigid_body_potential(mesh, :yaw)
edge_velocity = body_relative_edge_velocity(mesh, forward_speed, 0.0, 0.0;
    surge_gradient = surge.potential_gradient,
    sway_gradient = sway.potential_gradient,
    yaw_gradient = yaw.potential_gradient)
topology = build_surface_topology(mesh)
metrics = build_surface_metrics(mesh, topology)
active, retained = attached_flow_domain(mesh, edge_velocity, kinematic_viscosity;
    topology, metrics, curvature_limit = 0.5, minimum_retained = 0.0)

sections = sectional_crossflow_geometry(grid, falses(mesh.nfaces))
excluded = findall(.!active)
separation_x = isempty(excluded) ? minimum(mesh.centers[:, 1]) :
               maximum(mesh.centers[excluded, 1])
shed = shed_vorticity_derivatives(sections, separation_x, forward_speed;
    rho = WATER_DENSITY, x_reference = midship)

correction = viscous_maneuvering_correction(grid, surge.potential_gradient,
    sway.potential_gradient, yaw.potential_gradient, forward_speed,
    kinematic_viscosity; rho = WATER_DENSITY, linearization = :central)
viscous = nondimensionalize_viscous_derivatives(correction.derivatives,
    ship_length, forward_speed; rho = WATER_DENSITY)

# Convert this package's Wang normalisation to the MMG one the table above uses.
to_mmg(wang) = abs(wang) / draft_ratio
shed_mmg = to_mmg(shed.Y_v / (WATER_DENSITY / 2 * ship_length^2 * forward_speed))
viscous_mmg = to_mmg(viscous.Y_v)
@printf("shed vorticity        Y'_V = %.4f\n", shed_mmg)
@printf("displacement+friction Y'_V = %.4f\n", viscous_mmg)
@printf("total                 Y'_V = %.4f\n", shed_mmg + viscous_mmg)

shed vorticity        Y'_V = 0.3747
displacement+friction Y'_V = 0.0614
total                 Y'_V = 0.4361


## The full comparison

In [4]:
total_mmg = shed_mmg + viscous_mmg
rows = [(labels[m], empirical_sway_derivatives(kvlcc2; method = m).Y_v)
        for m in empirical_methods()]
push!(rows, ("This package (BL + shed vorticity)", total_mmg))
push!(rows, ("RANS (Chame et al. 2025)", chame_cfd))

println(rpad("method", 36), rpad("Y'_V", 9), rpad("vs Chame EFD", 14), "vs MMG set")
println(repeat("-", 74))
for (name, value) in rows
    @printf("%-36s%-9.4f%+-14.1f%+.1f%%\n", name, value,
        100 * (value / chame_measured - 1), 100 * (value / mmg_measured - 1))
end
println(repeat("-", 74))
@printf("%-36s%-9.4f\n", "EFD (Chame et al. 2025)", chame_measured)
@printf("%-36s%-9.4f\n", "EFD (MMG set)", mmg_measured)

method                              Y'_V     vs Chame EFD  vs MMG set
--------------------------------------------------------------------------
Low aspect ratio (theory)           0.2042   -24.6         -35.2%
Jacobs (1966)                       0.2042   -24.6         -35.2%
Smitt (1970)                        0.3247   +19.8         +3.1%
Norrbin (1971)                      0.3568   +31.7         +13.3%
Inoue et al. (1981)                 0.4097   +51.2         +30.1%
Clarke, Gedling & Hine (1982)       0.3886   +43.4         +23.4%
Ho-Young Lee et al. (1998)          0.3077   +13.6         -2.3%
This package (BL + shed vorticity)  0.4361   +60.9         +38.4%
RANS (Chame et al. 2025)            0.2760   +1.8          -12.4%
--------------------------------------------------------------------------
EFD (Chame et al. 2025)             0.2710   
EFD (MMG set)                       0.3150   


### Reading this honestly

The method lands **essentially on top of Clarke** — and both over-predict. That is
worth dwelling on, because it is informative in a way that a lucky agreement
would not be.

The shed-vorticity term alone is close to $2\pi(T/L)^2$ in this package's
normalisation, which is $2\pi T/L$ in MMG terms, i.e. **twice the classical
low-aspect-ratio value** of $\pi T/L$ that Chame's Table 1 and Gokarn both quote.
The factor is a real and unresolved discrepancy, not a rounding matter, and it
comes down to how much of the sectional momentum is taken to be lost at
separation:

- The momentum balance as implemented uses the **double body's** sectional added
  mass $\rho\pi T^2$ and assumes all of it is shed. That choice is what makes the
  closed-body limit reproduce the boundary-element Munk moment to within 31%,
  the expected slender-body error for a hull this blunt; halving it gives 66% of
  the Munk moment, which is clearly wrong.
- The classical low-aspect-ratio value corresponds to shedding **half** of it.

So the two constraints pull opposite ways: the Munk-moment check supports the
larger value, the textbook low-aspect-ratio result supports the smaller. The
measurement sits between them. This is stated rather than tuned away, and it is
the first thing to resolve if the model is developed further.

In [5]:
low_aspect = empirical_sway_derivatives(kvlcc2;
    method = :low_aspect_ratio).Y_v
println("bracketing the measurement")
@printf("  half the shed term (classical low-aspect-ratio) %.4f\n",
    shed_mmg / 2 + viscous_mmg)
@printf("  EFD (Chame)                                     %.4f\n", chame_measured)
@printf("  EFD (MMG set)                                   %.4f\n", mmg_measured)
@printf("  full shed term (as implemented)                 %.4f\n", total_mmg)
@printf("\nbare low-aspect-ratio theory, no viscous term:   %.4f\n", low_aspect)
@printf("ratio of shed term to low-aspect-ratio theory:   %.2f\n",
    shed_mmg / low_aspect)

bracketing the measurement
  half the shed term (classical low-aspect-ratio) 0.2487
  EFD (Chame)                                     0.2710
  EFD (MMG set)                                   0.3150
  full shed term (as implemented)                 0.4361

bare low-aspect-ratio theory, no viscous term:   0.2042
ratio of shed term to low-aspect-ratio theory:   1.83


## What each method actually knows

The numbers above flatten a real distinction. These methods do not all take the
same inputs, and that governs what they can be used for.

| method | inputs | can it see a hull-form change? |
|---|---|---|
| Low aspect ratio | $T$, $L$ | only through draft and length |
| Jacobs | $T$, $L$, lateral area, centroid | partly |
| Smitt, Norrbin, Inoue, Clarke | $L, B, T, C_B$ | only through four numbers |
| Ho-Young Lee, Tae Lee | $L, B, T, C_B$, stern layout | a little more |
| Chame et al. (2025) | as above plus bow and stern shape terms | more |
| **This package** | **the hull surface mesh** | **yes — any change of shape** |

That last row is the reason for the whole exercise. A regression cannot respond
to a change in the shape of the bilge, or to the flare of the bow, because those
never enter its arguments. A method built on the mesh can, and can be
differentiated with respect to it — which is what makes hull-form optimisation
possible rather than just hull-form ranking.

## Jacobs, and why the older methods sit low

Jacobs (1966) computes $Y'_V$ from slender-body theory with a correction for the
lateral area distribution, and comes out **below** the measurement — as does the
bare low-aspect-ratio theory it extends. Chame et al. report $-18.8\%$ for Jacobs
on this hull.

The reason is physical and is the same one notebook 05 is built around: slender
body theory returns the lateral momentum to the flow at the tail unless something
takes it away. Jacobs' correction is for the *distribution* of lateral area, not
for separation. Methods that do not model the flow leaving the hull under-predict
$Y_v$; methods fitted to data absorb it into their coefficients and, when applied
to a hull unlike their database, over-shoot. Both failure modes are visible in
the table above.

## Yaw, and the acceleration derivatives

$Y_v$ has been the focus because it is the one ideal flow gets completely wrong.
The others are less dramatic: $N_v$ is dominated by the Munk moment, which
potential flow gives exactly, and the acceleration derivatives are added-mass
quantities that a boundary-element solve computes directly.

Clarke's full set is included below for completeness, since it covers all eight.

In [6]:
clarke = clarke_full_derivatives(kvlcc2)
hirano = hirano_takashina_derivatives(kvlcc2)
println("Clarke, Gedling & Hine (1982), full set, MMG normalisation")
for name in (:Y_vdot, :Y_v, :N_vdot, :N_v, :Y_rdot, :Y_r, :N_rdot, :N_r)
    @printf("  %-8s %+.5f\n", name, getproperty(clarke, name))
end
println("\nHirano & Takashina (2010), low aspect ratio")
for name in (:Y_v, :N_v, :Y_r, :N_r)
    @printf("  %-8s %+.5f\n", name, getproperty(hirano, name))
end

Clarke, Gedling & Hine (1982), full set, MMG normalisation


  Y_vdot   -0.01584
  Y_v      -0.02526
  N_vdot   -0.00113
  N_v      -0.00871
  Y_rdot   -0.00127
  Y_r      +0.00864
  N_rdot   -0.00082
  N_r      -0.00314

Hirano & Takashina (2010), low aspect ratio
  Y_v      +0.00128
  N_v      -0.13000
  Y_r      +0.10210
  N_r      -0.05330


Note the Hirano and Takashina $Y'_v$: on a hull as full as KVLCC2 its
$+1.4\,C_B B/L$ correction very nearly cancels the $-\tfrac12\pi\Lambda$ term and
the result comes out marginally **positive**, which no real hull does. The
formula is outside its useful range at $C_B\approx0.81$. This is asserted in the
test suite rather than left for a reader to trip over.

## The drift-angle limit

Chame et al. establish something the linear derivative alone cannot express: the
drift angle beyond which linear theory stops describing the force at all. They
set it at **10° for blunt hulls and 8° for slender hulls**.

That is directly checkable here, because the virtual captive test sweeps drift
and fits both a linear and a cubic term. If linearity fails around 8 to 10°, the
cubic contribution should become comparable to the linear one there.

The sweep is capped at 8 degrees, and not by choice: past about 9
degrees the quasi-3D march refuses to run at all, raising `Head's shape factor
H must exceed 1.1`. The attached-flow closure has no solution there. That the
boundary layer model gives out at close to the drift angle Chame et al. identify
as the end of linear validity is worth noticing, though it is also simply a hard
limit on how far this sweep can go.

In [7]:
captive = virtual_captive_test(grid, surge.potential_gradient,
    sway.potential_gradient, yaw.potential_gradient, forward_speed,
    kinematic_viscosity; drift_angles = range(-0.14, 0.14; length = 9),
    yaw_rates = [0.0], rho = WATER_DENSITY, x_reference = midship)
linear = captive.coefficients.sway.Y_v
cubic = captive.coefficients.sway.Y_vvv
@printf("Y_v' = %+.5g   Y_vvv' = %+.5g\n", linear, cubic)
println("\ncubic share of the modelled sway force against drift angle")
for degrees in (2.0, 4.0, 6.0, 8.0)
    v = -sin(deg2rad(degrees))
    linear_part = linear * v
    cubic_part = cubic * v^3
    @printf("  %4.1f deg   cubic/linear = %5.1f%%\n", degrees,
        100 * abs(cubic_part / linear_part))
end

Y_v' = -0.0016954   Y_vvv' = +0.020773

cubic share of the modelled sway force against drift angle
   2.0 deg   cubic/linear =   1.5%
   4.0 deg   cubic/linear =   6.0%
   6.0 deg   cubic/linear =  13.4%
   8.0 deg   cubic/linear =  23.7%


## Summary

- **Two experimental values for KVLCC2 differ by 16%.** Nothing here resolves
  better than that.
- **The classical regressions split.** Pre-1980 methods and bare slender-body
  theory under-predict; Norrbin, Inoue and Clarke over-predict by 30 to 50%.
  Chame et al.'s finding, that these formulas do not transfer to modern hulls,
  is reproduced here directly.
- **This package's method lands on Clarke**, without being fitted to anything —
  and inherits Clarke's over-prediction on this hull.
- **A factor-of-two ambiguity in the shed term is unresolved.** The Munk-moment
  check supports the value as implemented; the textbook low-aspect-ratio result
  supports half of it. The measurement lies between. This is the single most
  worthwhile thing to settle next.
- **Only the mesh-based method can respond to hull shape** beyond four
  parameters, and only it can be differentiated with respect to the geometry.
  That, rather than accuracy on one hull, is the case for it.